# 基于金庸小说全集的命名实体嵌入建模与降维可视化

#### 人物筛选：从15部小说中提取133个关键人物（见personlist.txt）

#### 文本预处理：加载停用词表，进行分词与清洗

#### 嵌入训练：使用Word2Vec生成300维词向量

#### 降维可视化：通过PCA将人物向量降至2D/3D空间

In [9]:
# 导入必要的库和模块，分别用于文件处理、文本分词、词向量计算、数据处理、可视化等任务
import os  # 操作系统相关功能，用于路径操作、文件管理等
import jieba  # 中文分词工具，用于文本分词处理
from gensim.models import Word2Vec  # Word2Vec模型，用于训练词向量
import numpy as np  # 数值计算库，用于矩阵操作和数值计算
from sklearn.decomposition import PCA  # PCA降维工具，用于数据降维
from sklearn.cluster import KMeans  # KMeans算法，用于聚类分析
import matplotlib.pyplot as plt  # 绘图库，用于数据可视化
from mpl_toolkits.mplot3d import Axes3D  # 3D绘图工具，用于绘制三维图形
import re  # 正则表达式模块，用于字符串模式匹配
from tqdm import tqdm  # 进度条库，用于显示进度
import pickle  # 用于序列化和反序列化对象的库
from collections import defaultdict  # defaultdict字典，用于提供缺失键的默认值
import chardet  # 字符编码检测库，用于自动检测文件编码


# 配置matplotlib的中文显示和负号显示
plt.rcParams['font.sans-serif'] = ['SimHei']  # 用来正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False  # 用来正常显示负号


# 加载停用词表，将文件中的停用词读取到一个集合中
def load_stop_words(file_path):
    """加载停用词表
    Args:
        file_path: 停用词表文件路径
    Returns:
        set: 停用词集合
    """
    # 打开指定路径的停用词文件，读取内容，默认以UTF-8编码
    with open(file_path, 'r', encoding='utf-8') as f:
        # 使用生成器表达式逐行读取文件，并去掉每行的前后空白字符，筛选出非空行，最后将这些词放入集合中
        stop_words = set(line.strip() for line in f if line.strip())  # 读取文件并构建停用词集合
    return stop_words  # 返回停用词集合

# 处理单个文本文件，返回一个句子列表，每个句子是词的列表
def process_text_file(file_path, stop_words, characters):
    """处理单个文本文件，返回句子列表
    Args:
        file_path: 文本文件路径
        stop_words: 停用词集合
        characters: 人物名称字典
    Returns:
        list: 句子列表，每个句子是词列表
    """
    # 将所有人物名称添加到jieba词典中，确保人物名称不会被错误分词
    for name in characters.keys():  # 遍历人物名称字典的所有键（人物名称）
        jieba.add_word(name)  # 将人物名称添加到jieba词典中
        if characters[name]:  # 如果该人物有别名
            jieba.add_word(characters[name])  # 将别名也添加到jieba词典中
    
    # 定义句子分隔符，表示句子的结束符号
    sentence_endings = ['。', '！', '？', '；', '……', '——', '\n', '…', '—']  # 句末分隔符（包括句号、问号等）
    sentence_continuations = ['，', '、', '：', '；', '（', '）', '"', '"', '「', '」', '『', '』']  # 句中分隔符（用于标识句子中继续的符号）
    sentences = []  # 初始化一个空列表，用于存储分词后的句子
    
    # 打开文件以读取内容，并自动检测文件编码
    with open(file_path, 'rb') as file:  # 以二进制模式打开文件，用于读取文件内容
        content = file.read()  # 读取文件内容
        encoding = chardet.detect(content)['encoding']  # 使用chardet库自动检测文件的编码方式
        
    # 使用检测到的编码格式重新打开文件并读取文本内容
    with open(file_path, 'r', encoding=encoding, errors='replace') as file:  # 以正确编码方式打开文件
        text = file.read()  # 读取文本内容

        # 清理文本，例如去除无关字符等
        text = clean_text(text)  # 调用clean_text函数清理文本

        # 使用jieba进行分词，将文本分割成词
        words = jieba.lcut(text)  # 使用jieba进行精确分词，返回分词后的词列表

        # 将文本按句子分割，句子是通过分隔符来定义的
        current_sentence = []  # 当前句子，初始化为空列表
        for word in words:  # 遍历分词后的词列表
            if word in sentence_endings:  # 如果当前词是句子的结束符
                if current_sentence:  # 如果当前句子不为空
                    sentences.append(current_sentence)  # 将当前句子添加到句子列表中
                    current_sentence = []  # 重置当前句子为空列表
            elif word in sentence_continuations:  # 如果当前词是句中的继续符
                if current_sentence:  # 如果当前句子不为空，则继续添加当前句子
                    current_sentence.append(word)  # 将词添加到当前句子中
            elif word not in stop_words:  # 只保留不是停用词的词
                current_sentence.append(word)  # 将非停用词添加到当前句子中
        
        # 处理最后一个句子，如果当前句子非空，则添加到句子列表
        if current_sentence:  # 如果当前句子非空
            sentences.append(current_sentence)  # 将当前句子添加到句子列表中

    # 过滤掉过短的句子，返回句子列表中长度大于等于3的句子
    return [s for s in sentences if len(s) >= 3]  # 只返回长度大于等于3的句子    


# 清理文本中的标点符号，规范标点符号的使用
def clean_text(text):
    """清理文本中的标点符号
    Args:
        text: 原始文本
    Returns:
        str: 清理后的文本
    """
    # 替换连续的标点符号为单个标点
    text = re.sub(r'[。！？；]+', '。', text)  # 将连续的句末标点（如'。'、'！'、'？'等）替换为单个句号
    text = re.sub(r'[，、；]+', '，', text)    # 将连续的句中标点（如'，'、'、'、'；'等）替换为单个逗号
    text = re.sub(r'[……]+', '……', text)      # 将连续的省略号（'……'）替换为标准的省略号
    text = re.sub(r'[——]+', '——', text)      # 将连续的破折号（'——'）替换为标准的破折号
    # 处理对话中的引号，将多余的引号替换为标准形式
    text = re.sub(r'[""]+', '"', text)  # 将多余的双引号替换为单个双引号
    text = re.sub(r'[「」]+', '「', text)  # 将多余的‘」’符号替换为标准的‘「’
    text = re.sub(r'[『』]+', '『', text)  # 将多余的‘』’符号替换为标准的‘『’
    return text  # 返回清理后的文本


# 加载人物列表，并处理人物名称和别名
def load_character_list(file_path):
    """加载人物列表，处理别名
    Args:
        file_path: 人物列表文件路径
    Returns:
        dict: 人物名称字典，键为主名，值为别名
    """
    characters = {}  # 初始化一个空字典，用来存储人物名称及其别名
    with open(file_path, 'r', encoding='utf-8') as f:  # 打开文件，使用UTF-8编码读取
        for line in f:  # 遍历文件中的每一行
            line = line.strip()  # 去除行首和行尾的空白字符
            if not line:  # 如果当前行为空，则跳过
                continue
            # 使用正则表达式匹配括号中的别名，格式为“主名（别名）”
            match = re.match(r'(.+?)（(.+?)）', line)  # 匹配主名和别名
            if match:  # 如果匹配成功
                main_name = match.group(1)  # 获取主名
                alias = match.group(2)  # 获取别名
                characters[main_name] = alias  # 将主名作为键，别名作为值存入字典
            else:
                characters[line] = None  # 如果没有别名，直接将人物名称作为键，值为None
    return characters  # 返回包含人物名称及其别名的字典


# 训练Word2Vec模型，生成词向量
def train_word2vec(sentences, vector_size=200, window=8, min_count=5, workers=4, epochs=10):
    """训练Word2Vec模型
    Args:
        sentences: 句子列表，每个句子是一个词的列表
        vector_size: 词向量的维度（默认200）
        window: 窗口大小（默认8）
        min_count: 最小词频，低于该频率的词将被忽略（默认5）
        workers: 用于训练的并行工作线程数（默认4）
        epochs: 训练的轮数（默认10）
    Returns:
        Word2Vec: 训练好的Word2Vec模型
    """
    # 创建并训练Word2Vec模型
    model = Word2Vec(
        sentences=sentences,  # 训练数据：每个句子是一个词列表
        vector_size=vector_size,  # 设定词向量的维度
        window=window,  # 设置上下文窗口的大小
        min_count=min_count,  # 设置最小词频，忽略低于该频率的词
        workers=workers,  # 设置训练时使用的线程数
        epochs=epochs,  # 设定训练的轮数
        sg=1,  # 使用skip-gram模型（默认是CBOW模型，设置sg=1切换为skip-gram模型）
        hs=0,  # 使用负采样（hs=0表示使用负采样，hs=1表示使用层次Softmax）
        negative=5,  # 负采样的数量，指定每次训练时负样本的数量
        ns_exponent=0.75  # 负采样指数，用于调整负采样的概率分布
    )
    return model  # 返回训练好的模型


# 获取人物向量，处理别名情况
def get_character_vector(model, main_name, alias=None):
    """获取人物向量，处理别名情况
    Args:
        model: 训练好的Word2Vec模型
        main_name: 主要名称，人物的主名
        alias: 别名，人物的可选别名
    Returns:
        np.ndarray: 人物的词向量，如果找不到该人物向量则返回None
    """
    vectors = []  # 用于存储人物名称的向量列表
    
    # 尝试获取主名的词向量
    try:
        vectors.append(model.wv[main_name])  # 获取主名称的词向量并加入列表
    except KeyError:
        pass  # 如果没有找到该词向量，则忽略该错误
    
    # 如果提供了别名，则尝试获取别名的词向量
    if alias:
        try:
            vectors.append(model.wv[alias])  # 获取别名的词向量并加入列表
        except KeyError:
            pass  # 如果没有找到别名的词向量，则忽略该错误
    
    # 如果没有找到任何词向量，则返回None
    if not vectors:
        return None
    
    # 如果找到了多个向量（主名和别名都有），返回它们的平均值
    return np.mean(vectors, axis=0)  # 计算多个向量的平均值并返回


# 绘制2D可视化图
def plot_2d(vectors, characters, clusters=None, save_path='characters_2d.png'):
    """绘制2D可视化图
    Args:
        vectors: 2D向量，每个人物的2D坐标
        characters: 人物名称列表
        clusters: 聚类结果，表示不同人物的类别
        save_path: 保存路径，图像保存的文件路径
    """
    
    # 创建一个15x15大小的图形，调整画布以使得图形更加紧凑
    plt.figure(figsize=(15, 15))  
    
    # 绘制散点图，如果提供了聚类信息，则根据聚类结果设置颜色
    if clusters is not None:
        # 根据聚类结果设置颜色，使用YlOrRd调色板，点的大小为100，透明度为0.7
        scatter = plt.scatter(vectors[:, 0], vectors[:, 1], c=clusters, cmap='YlOrRd', s=100, alpha=0.7)
    else:
        # 如果没有聚类信息，则只绘制普通的散点图
        plt.scatter(vectors[:, 0], vectors[:, 1], s=100, alpha=0.7)
    
    # 为每个点添加标签，使用透明度和字体设置来优化显示
    for i, char in enumerate(characters):
        plt.text(
            vectors[i, 0],  # x轴坐标
            vectors[i, 1],  # y轴坐标
            char,           # 人物名称
            fontsize=10,    # 设置字体大小
            alpha=0.8,      # 设置透明度
            ha='center',    # 设置水平对齐方式为居中
            va='center',    # 设置垂直对齐方式为居中
            fontweight='light'  # 设置字体为轻体
        )
    
    # 设置图形标题，使用SimHei字体以支持中文，标题字体加粗
    plt.title('金庸小说人物关系2D可视化展示', fontsize=50, fontproperties='SimHei', fontweight='bold')  
    # 设置坐标轴的字体大小
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    
    # 启用网格线，并设置网格线样式和透明度
    plt.grid(True, linestyle='--', alpha=0.3)

    # 设置坐标轴的显示范围，稍微扩大范围以避免点被遮挡
    plt.xlim(min(vectors[:, 0]) - 1, max(vectors[:, 0]) + 1)
    plt.ylim(min(vectors[:, 1]) - 1, max(vectors[:, 1]) + 1)

    # 将绘制的图像保存到指定的文件路径，DPI设置为300，确保高分辨率
    plt.savefig(save_path, dpi=300, bbox_inches='tight', format='png')
    plt.close()  # 关闭当前图形，以释放内存


# 绘制3D可视化图
def plot_3d(vectors, characters, clusters=None, save_path='characters_3d.png'):
    """绘制3D可视化图
    Args:
        vectors: 3D向量，每个人物的3D坐标
        characters: 人物名称列表
        clusters: 聚类结果，表示不同人物的类别
        save_path: 保存路径，图像保存的文件路径
    """
    
    # 创建一个15x15大小的图形，使图形更紧凑
    fig = plt.figure(figsize=(15, 15))  
    # 创建3D坐标轴
    ax = fig.add_subplot(111, projection='3d')
    
    # 绘制散点图
    if clusters is not None:
        # 如果有聚类结果，使用颜色区分不同的聚类
        scatter = ax.scatter(vectors[:, 0], vectors[:, 1], vectors[:, 2], c=clusters, cmap='YlOrRd', s=100, alpha=0.7)
    else:
        # 如果没有聚类结果，则绘制普通的3D散点图
        ax.scatter(vectors[:, 0], vectors[:, 1], vectors[:, 2], s=100, alpha=0.7)
    
    # 为每个点添加标签，使用透明度和字体设置来优化显示
    for i, char in enumerate(characters):
        ax.text(
            vectors[i, 0],  # x轴坐标
            vectors[i, 1],  # y轴坐标
            vectors[i, 2],  # z轴坐标
            char,           # 人物名称
            fontsize=10,    # 设置字体大小
            alpha=0.8,      # 设置透明度
            ha='center',    # 设置水平对齐方式为居中
            va='center',    # 设置垂直对齐方式为居中
            fontweight='light'  # 设置字体为轻体
        )
    
    # 设置图形标题，使用SimHei字体以支持中文，标题字体加粗
    ax.set_title('金庸小说人物关系3D可视化展示', fontsize=50, fontproperties='SimHei', fontweight='bold')  
    # 设置X、Y、Z轴的标题和字体大小
    ax.set_xlabel('X轴', fontsize=12)
    ax.set_ylabel('Y轴', fontsize=12)
    ax.set_zlabel('Z轴', fontsize=12)
    ax.tick_params(axis='both', which='major', labelsize=12)
    
    # 调整视角，使图形展示更加清晰
    ax.view_init(elev=25, azim=45)  # 调整俯仰角（elev）和方位角（azim）
    
    # 保存图片，确保图像分辨率高（DPI设置为300）
    plt.savefig(save_path, dpi=300, bbox_inches='tight', format='png')
    plt.close()  # 关闭图形，释放内存


# 主函数：执行文本处理、Word2Vec训练、聚类分析以及可视化
def main():
    
    # 加载停用词表
    print("正在加载停用词表...")
    stop_words = load_stop_words('stop_words.txt')  # 加载停用词
    
    # 加载人物列表
    print("正在加载人物列表...")
    characters = load_character_list('personlist.txt')  # 加载人物和别名字典
    
    # 处理文本文件
    print("正在处理文本文件...")
    all_sentences = []
    folder_path = '金庸小说全集三联版txt'  # 指定小说文本所在的文件夹路径
    
    # 遍历文件夹中的每个文本文件
    for filename in tqdm(os.listdir(folder_path)):
        if filename.endswith('.txt'):  # 仅处理.txt文件
            file_path = os.path.join(folder_path, filename)
            sentences = process_text_file(file_path, stop_words, characters)  # 处理每个文件，获取句子列表
            all_sentences.extend(sentences)  # 将所有句子添加到总句子列表中
    
    # 训练Word2Vec模型
    print("正在训练Word2Vec模型...")
    model = train_word2vec(all_sentences)  # 使用处理后的句子训练Word2Vec模型
    
    # 保存训练好的模型
    model.save('jinyong_word2vec.model')  # 保存Word2Vec模型
    
    # 获取人物向量
    print("正在获取人物向量...")
    character_vectors = []  # 用于存储人物的词向量
    valid_characters = []  # 用于存储有效的人物名称
    
    # 遍历人物字典，获取每个人物的词向量
    for main_name, alias in tqdm(characters.items()):
        vector = get_character_vector(model, main_name, alias)  # 获取人物的词向量
        if vector is not None:  # 如果找到了词向量
            character_vectors.append(vector)  # 添加该人物的词向量到列表中
            valid_characters.append(main_name)  # 记录有效的人物名称
        else:
            print(f"警告：找不到人物 '{main_name}' 的向量")  # 如果找不到该人物的词向量，发出警告
    
    character_vectors = np.array(character_vectors)  # 转换为NumPy数组，方便后续处理
    
    # 进行聚类分析
    print("正在进行聚类分析...")
    n_clusters = min(20, len(valid_characters) // 2)  # 根据有效人物数量动态确定聚类数，最多20个
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)  # 使用KMeans算法进行聚类
    clusters = kmeans.fit_predict(character_vectors)  # 获取聚类结果
    
    # 2D PCA降维
    print("\n正在进行2D PCA降维...")
    pca_2d = PCA(n_components=2)  # 初始化PCA，设置降维到2D
    vectors_2d = pca_2d.fit_transform(character_vectors)  # 对人物词向量进行2D降维
    plot_2d(vectors_2d, valid_characters, clusters, 'characters_2d.png')  # 绘制并保存2D可视化图
    
    # 3D PCA降维
    print("正在进行3D PCA降维...")
    pca_3d = PCA(n_components=3)  # 初始化PCA，设置降维到3D
    vectors_3d = pca_3d.fit_transform(character_vectors)  # 对人物词向量进行3D降维
    plot_3d(vectors_3d, valid_characters, clusters, 'characters_3d.png')  # 绘制并保存3D可视化图
    
    # 保存聚类结果
    cluster_results = defaultdict(list)  # 用于存储聚类结果，字典的键是聚类编号，值是人物名称列表
    for char, cluster in zip(valid_characters, clusters):  # 将每个人物及其对应的聚类编号添加到结果中
        cluster_results[cluster].append(char)
    
    with open('cluster_results.txt', 'w', encoding='utf-8') as f:  # 保存聚类结果到文件
        for cluster, chars in cluster_results.items():
            f.write(f"聚类 {cluster}:\n")
            f.write(", ".join(chars) + "\n\n")
    
    print("\n处理完成！已生成2D和3D可视化文件，以及聚类结果。")  # 处理完成的提示


# 程序入口
if __name__ == "__main__":
    main()  # 调用主函数执行程序

正在加载停用词表...
正在加载人物列表...
正在处理文本文件...


100%|██████████| 15/15 [01:06<00:00,  4.45s/it]


正在训练Word2Vec模型...
正在获取人物向量...


100%|██████████| 138/138 [00:00<00:00, 23003.50it/s]

警告：找不到人物 '程青霜' 的向量
警告：找不到人物 '风一鸣' 的向量
警告：找不到人物 '牛头陀' 的向量
警告：找不到人物 '任盈盈' 的向量
警告：找不到人物 '新月公主' 的向量
警告：找不到人物 '易容者' 的向量
警告：找不到人物 '扫地僧' 的向量
正在进行聚类分析...

正在进行2D PCA降维...


正在进行3D PCA降维...

处理完成！已生成2D和3D可视化文件，以及聚类结果。
